In [147]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

In [148]:
data = pd.read_csv('data.csv')

In [149]:
data.head()

,CustomerID,Name,Age,Gender,Location,Email,Phone,Address,Segment,PurchaseHistory,SubscriptionDetails,ServiceInteractions,PaymentHistory,WebsiteUsage,ClickstreamData,EngagementMetrics,Feedback,MarketingCommunication,NPS,ChurnLabel,Timestamp
0,1001,Mark Barrett,31,Male,Andrewfort,allison74@example.net,3192528777,"61234 Shelley Heights Suite 467\nCohentown, GU...",Segment B,"[{'Product': 'Frozen Cocktail Mixes', 'Frequen...","{'Plan': 'Express', 'Start_Date': '2020-06-08'...","[{'Type': 'Call', 'Date': '2019-09-26'}, {'Typ...","[{'Method': 'Credit Card', 'Late_Payments': 5}...","{'PageViews': 49, 'TimeSpent(minutes)': 15}","[{'Action': 'Add to Cart', 'Page': 'register',...","{'Logins': 19, 'Frequency': 'Weekly'}","{'Rating': 1, 'Comment': 'I move baby go small...","[{'Email_Sent': '2019-10-17', 'Email_Opened': ...",3,1,2020-01-27 01:36:49
1,1002,Jeremy Welch,66,Female,Millerhaven,fmiller@example.com,231-587-1818x8651,"4959 Jennifer Junction\nNew Angelaport, TN 87397",Segment C,"[{'Product': 'Watercraft Polishes', 'Frequency...","{'Plan': 'Pro', 'Start_Date': '2021-07-21', 'E...","[{'Type': 'Call', 'Date': '2020-01-05'}, {'Typ...","[{'Method': 'Credit Card', 'Late_Payments': 3}...","{'PageViews': 100, 'TimeSpent(minutes)': 9}","[{'Action': 'Add to Cart', 'Page': 'homepage',...","{'Logins': 9, 'Frequency': 'Weekly'}","{'Rating': 2, 'Comment': 'Wish what bag cut li...","[{'Email_Sent': '2021-08-02', 'Email_Opened': ...",6,0,2019-01-06 18:30:03
2,1003,Brandon Patel,36,Female,Lozanostad,jasonbrown@example.org,(270)633-9095,"38701 Amanda Brook Apt. 076\nKimshire, NJ 62516",Segment B,"[{'Product': 'Vehicle Waxes, Polishes & Protec...","{'Plan': 'Essential', 'Start_Date': '2019-10-0...","[{'Type': 'Email', 'Date': '2019-10-09'}, {'Ty...","[{'Method': 'Credit Card', 'Late_Payments': 1}...","{'PageViews': 1, 'TimeSpent(minutes)': 97}","[{'Action': 'Search', 'Page': 'terms', 'Timest...","{'Logins': 19, 'Frequency': 'Monthly'}","{'Rating': 4, 'Comment': 'Some Democrat guess ...","[{'Email_Sent': '2021-08-29', 'Email_Opened': ...",3,0,2019-04-30 04:25:10
3,1004,Tina Martin,62,Female,South Dustin,matthew62@example.net,050.081.8706x11982,"67324 Ashley Coves\nSouth John, RI 29650",Segment C,"[{'Product': 'Mouthwash', 'Frequency': 5, 'Val...","{'Plan': 'Smart', 'Start_Date': '2020-01-14', ...","[{'Type': 'Call', 'Date': '2020-08-28'}, {'Typ...","[{'Method': 'Credit Card', 'Late_Payments': 36...","{'PageViews': 25, 'TimeSpent(minutes)': 31}","[{'Action': 'Click', 'Page': 'privacy', 'Times...","{'Logins': 4, 'Frequency': 'Daily'}","{'Rating': 1, 'Comment': 'Yard feel never miss...","[{'Email_Sent': '2021-02-03', 'Email_Opened': ...",1,1,2020-03-03 17:33:28
4,1005,Christopher Rodriguez,68,Female,West James,shannonstrickland@example.org,+1-701-854-4915x724,"01169 Miller Mission\nWest Anthonyburgh, WY 47359",Segment C,"[{'Product': 'Ice Cream Novelties', 'Frequency...","{'Plan': 'Basic', 'Start_Date': '2021-04-08', ...","[{'Type': 'Call', 'Date': '2019-04-10'}, {'Typ...","[{'Method': 'Credit Card', 'Late_Payments': 0}...","{'PageViews': 77, 'TimeSpent(minutes)': 51}","[{'Action': 'Click', 'Page': 'privacy', 'Times...","{'Logins': 12, 'Frequency': 'Weekly'}","{'Rating': 3, 'Comment': 'Ten determine unit i...","[{'Email_Sent': '2022-03-11', 'Email_Opened': ...",3,0,2019-04-05 22:42:22


In [150]:
cols_to_extract = ['PurchaseHistory', 'SubscriptionDetails', 'ServiceInteractions', 'PaymentHistory', 
                   'WebsiteUsage', 'ClickstreamData', 'EngagementMetrics', 'Feedback', 'MarketingCommunication']

In [151]:
# I want to see the cell of the first row in the ServiceInteractions column
data['ServiceInteractions'].iloc[0]

"[{'Type': 'Call', 'Date': '2019-09-26'}, {'Type': 'Chat', 'Date': '2021-07-25'}, {'Type': 'Email', 'Date': '2020-04-13'}, {'Type': 'Chat', 'Date': '2020-11-15'}]"

In [152]:
import ast  # Used to safely convert string representations of lists/dicts into real Python objects

# Converts a string like "[{'a': 1}]" into an actual list/dict
def convert_string(col: str) -> any:
    # If the value is a string, try to interpret it as Python data
    if isinstance(col, str): 
        return ast.literal_eval(col)
    # If it's already a list or dict, leave it alone
    return col

# Dictionary to store the extracted / normalized tables
# Key = column name, Value = cleaned DataFrame
extracted_tables = {}

# Loop through each column that contains nested data
for col in cols_to_extract:
    
    # Convert stringified JSON-like values into real Python objects
    data[col] = data[col].apply(convert_string)
    
    # Placeholder DataFrame for the normalized output
    normalized = pd.DataFrame()

    # CASE 1: Column contains a LIST of dictionaries (most common case)
    if isinstance(data[col].iloc[0], list):
        
        # Explode turns each list item into its own row
        # CustomerID is duplicated so relationships are preserved
        temp = data[['CustomerID', col]].explode(col)
        
        # Convert dictionaries into flat columns (e.g. Type, Date)
        normalized = pd.json_normalize(temp[col])
        
        # Reattach CustomerID to each expanded row
        normalized['CustomerID'] = temp['CustomerID'].values

    # CASE 2: Column contains a single dictionary per row
    elif isinstance(data[col].iloc[0], dict):
        
        # Normalize dictionary into columns
        normalized = pd.json_normalize(data[col])
        
        # Attach CustomerID so rows stay linked to customers
        normalized['CustomerID'] = data['CustomerID'].values

    # Store the processed DataFrame for this column
    extracted_tables[col] = normalized


In [153]:
extracted_tables.keys()

dict_keys(['PurchaseHistory', 'SubscriptionDetails', 'ServiceInteractions', 'PaymentHistory', 'WebsiteUsage', 'ClickstreamData', 'EngagementMetrics', 'Feedback', 'MarketingCommunication'])

In [154]:
# Assign each extracted DataFrame from the dictionary to a separate variable
# extracted_tables is a dict: keys are table names, values are DataFrames
# FYI \ is a line continuation character in Python
PurchaseHistory, SubscriptionDetails, ServiceInteractions, PaymentHistory, WebsiteUsage, \
ClickstreamData, EngagementMetrics, Feedback, MarketingCommunication = \
[extracted_tables[k] for k in extracted_tables.keys()]


In [155]:
for key, values in extracted_tables.items():
    print(key, values.shape)

PurchaseHistory (68628, 4)
SubscriptionDetails (12483, 4)
ServiceInteractions (254253, 3)
PaymentHistory (37449, 3)
WebsiteUsage (12483, 3)
ClickstreamData (319616, 4)
EngagementMetrics (12483, 3)
Feedback (12483, 3)
MarketingCommunication (68762, 4)


## Feature Engineering

Purchase Behavior Features

**Idea:** Customers who spend more and buy consistently are less likely to churn.

We calculate:
- Total/average spending (loyalty indicator)
- Purchase consistency (stable vs erratic buyers)
- Product diversity (engaged vs single-product users)

In [156]:
# Group purchase history by each customer
PurchaseHistory = PurchaseHistory.groupby('CustomerID').agg({

    # Sum total money spent by the customer
    'Value': 'sum',

    # Sum total number of purchases made by the customer
    'Frequency': 'sum',

    # Count how many unique products the customer bought
    'Product': 'nunique'

}).reset_index()  # Turn CustomerID back into a normal column


# Rename columns to clearer, feature-style names
PurchaseHistory.columns = [
    'CustomerID',
    'total_purchase_value',
    'total_frequency',
    'product_diversity'
]

In [157]:
PurchaseHistory

,CustomerID,total_purchase_value,total_frequency,product_diversity
0,1001,3994.72,38,7
1,1002,2844.35,4,3
2,1003,1866.52,14,3
3,1004,1378.64,28,5
4,1005,2425.05,39,6
...,...,...,...,...
12478,13479,1196.56,14,3
12479,13480,710.57,1,1
12480,13481,5154.42,63,10
12481,13482,6055.16,58,9


### Subscription Lifecycle Features

**Idea:** Long-term subscribers with active accounts are less likely to churn.

We calculate:
- How long they've been subscribed (tenure)
- Whether subscription is still active
- Subscription plan type

In [158]:
# Convert subscription start date from string to datetime 
SubscriptionDetails['Start_Date'] = pd.to_datetime(SubscriptionDetails['Start_Date'])

# Convert subscription end date from string to datetime
SubscriptionDetails['End_Date'] = pd.to_datetime(SubscriptionDetails['End_Date'])


# Calculate total length of the subscription in days
SubscriptionDetails['subscription_duration_days'] = (
    SubscriptionDetails['End_Date'] - SubscriptionDetails['Start_Date']
).dt.days


# Calculate how old the subscription is (from start date to reference date)
SubscriptionDetails['subscription_age_days'] = (
    pd.to_datetime('2023-01-01') - SubscriptionDetails['Start_Date']
).dt.days


# Create a binary flag:
# 1 = subscription is still active on 2023-01-01
# 0 = subscription has ended before that date
SubscriptionDetails['is_active'] = (
    SubscriptionDetails['End_Date'] > pd.to_datetime('2023-01-01')
).astype(int)


# Keep only the columns needed for modeling / analysis
SubscriptionDetails = SubscriptionDetails[
    [
        'CustomerID',
        'Plan',
        'subscription_duration_days',
        'subscription_age_days',
        'is_active'
    ]
]

In [159]:
SubscriptionDetails

,CustomerID,Plan,subscription_duration_days,subscription_age_days,is_active
0,1001,Express,871,937,0
1,1002,Pro,290,529,0
2,1003,Essential,319,1184,0
3,1004,Smart,803,1083,0
4,1005,Basic,580,633,0
...,...,...,...,...,...
12478,13479,Essential,745,1296,0
12479,13480,Flex,18,22,0
12480,13481,Deluxe,20,546,0
12481,13482,Gold,484,894,0


### Service Interaction Patterns

**Idea:** Frequent support contacts (especially calls) may signal dissatisfaction.

We calculate:
- Total interactions
- Calculate last interaction date

In [160]:
# Convert the interaction date column from string to datetime
ServiceInteractions['Date'] = pd.to_datetime(ServiceInteractions['Date'])


# Aggregate interaction data per customer
ServiceInteractions = ServiceInteractions.groupby('CustomerID').agg({

    # Get the most recent interaction date per customer
    'Date': 'max',

    # Count total number of interactions per customer
    # (counts rows → each row = one interaction)
    'Type': 'count'

}).reset_index()


# Rename columns to be clear and model-friendly
ServiceInteractions.columns = [
    'CustomerID',
    'last_interaction_date',
    'total_interactions'
]

In [161]:
ServiceInteractions

,CustomerID,last_interaction_date,total_interactions
0,1001,2021-07-25,4
1,1002,2022-12-13,19
2,1003,2022-01-04,3
3,1004,2022-11-10,59
4,1005,2022-12-19,10
...,...,...,...
12478,13479,2022-10-09,10
12479,13480,2022-11-05,3
12480,13481,2022-12-08,26
12481,13482,2022-11-15,13


### Payment Reliability Features

**Idea:** Late payments indicate financial stress or dissatisfaction.

We calculate:
- Total late payments
- Late payment rate (percentage of payments that are late)
- Payment risk score (combined metric)

In [162]:
# Aggregate payment data at the customer level
PaymentHistory = PaymentHistory.groupby('CustomerID').agg({

    # Total number of late payments made by the customer
    'Late_Payments': 'sum',

    # Total number of payments made by the customer
    # (counting rows, each row = one payment)
    'Method': 'count'

}).reset_index()


# Rename columns for clarity and downstream use
PaymentHistory.columns = [
    'CustomerID',
    'total_late_payments',
    'payment_count'
]


# Calculate the proportion of payments that were late
# +1 prevents division by zero for customers with zero payments
PaymentHistory['late_payment_rate'] = (
    PaymentHistory['total_late_payments'] /
    (PaymentHistory['payment_count'] + 1)
)


# Create a composite risk score:
# customers with many late payments AND a high late rate score higher
PaymentHistory['payment_risk_score'] = (
    PaymentHistory['total_late_payments'] *
    PaymentHistory['late_payment_rate']
)

In [163]:
PaymentHistory

,CustomerID,total_late_payments,payment_count,late_payment_rate,payment_risk_score
0,1001,40,3,10.00,400.00
1,1002,10,3,2.50,25.00
2,1003,8,3,2.00,16.00
3,1004,79,3,19.75,1560.25
4,1005,2,3,0.50,1.00
...,...,...,...,...,...
12478,13479,3,3,0.75,2.25
12479,13480,6,3,1.50,9.00
12480,13481,83,3,20.75,1722.25
12481,13482,67,3,16.75,1122.25


### Digital Engagement Features

**Idea:** Active website users are more engaged and less likely to leave.

We calculate:
- Page views and time spent
- Engagement ratio (time per page)
- Engagement intensity (overall activity level)

In [164]:
# Aggregate website activity per customer
WebsiteUsage = WebsiteUsage.groupby('CustomerID').agg({

    # Total number of pages viewed by the customer
    'PageViews': 'sum',

    # Total time spent on the website (in minutes)
    'TimeSpent(minutes)': 'sum'

}).reset_index()


# Average time spent per page view
# +1 avoids division by zero when PageViews = 0
WebsiteUsage['engagement_ratio'] = (
    WebsiteUsage['TimeSpent(minutes)'] /
    (WebsiteUsage['PageViews'] + 1)
)


# Combined engagement strength:
# high only when BOTH views and time are high
WebsiteUsage['engagement_intensity'] = (
    WebsiteUsage['PageViews'] *
    WebsiteUsage['TimeSpent(minutes)']
)

In [165]:
WebsiteUsage

,CustomerID,PageViews,TimeSpent(minutes),engagement_ratio,engagement_intensity
0,1001,49,15,0.300000,735
1,1002,100,9,0.089109,900
2,1003,1,97,48.500000,97
3,1004,25,31,1.192308,775
4,1005,77,51,0.653846,3927
...,...,...,...,...,...
12478,13479,70,57,0.802817,3990
12479,13480,71,66,0.916667,4686
12480,13481,96,1,0.010309,96
12481,13482,63,2,0.031250,126


### Clickstream Behavior Features

**Idea:** Specific actions (adding to cart, searching) show purchase intent.

We calculate:
- Total actions taken
- Cart conversion rate (add-to-cart actions)
- Search intensity (exploration behavior)
- Page diversity (breadth of interest)

In [166]:
plus = lambda x: x + 1

plus(5)

6

In [167]:
# Aggregate clickstream behavior per customer
ClickstreamData = ClickstreamData.groupby('CustomerID').agg({

    # Count total number of actions performed by the customer
    'Action': [
        'count',                                  # total actions
        lambda x: (x == 'Add to Cart').sum(),     # number of add-to-cart actions
        lambda x: (x == 'Search').sum(),          # number of search actions
        lambda x: (x == 'Click').sum()            # number of click actions
    ],

    # Count number of unique pages visited
    'Page': 'nunique'

}).reset_index()


# Flatten multi-level column names into readable feature names
ClickstreamData.columns = [
    'CustomerID',
    'total_actions',
    'add_to_cart_count',
    'search_count',
    'click_count',
    'unique_pages'
]


# Proportion of actions that lead to adding items to the cart
# +1 avoids division by zero
ClickstreamData['cart_conversion_rate'] = (
    ClickstreamData['add_to_cart_count'] /
    (ClickstreamData['total_actions'] + 1)
)


# Proportion of actions that are searches
# Indicates exploratory vs goal-driven behavior
ClickstreamData['search_intensity'] = (
    ClickstreamData['search_count'] /
    (ClickstreamData['total_actions'] + 1)
)


# Diversity of pages visited relative to actions taken
# Measures how spread-out the user’s browsing is
ClickstreamData['page_diversity'] = (
    ClickstreamData['unique_pages'] /
    (ClickstreamData['total_actions'] + 1)
)

In [168]:
ClickstreamData

,CustomerID,total_actions,add_to_cart_count,search_count,click_count,unique_pages,cart_conversion_rate,search_intensity,page_diversity
0,1001,24,8,12,4,13,0.320000,0.480000,0.520000
1,1002,24,8,7,9,13,0.320000,0.280000,0.520000
2,1003,12,2,7,3,7,0.153846,0.538462,0.538462
3,1004,47,15,16,16,14,0.312500,0.333333,0.291667
4,1005,30,17,4,9,12,0.548387,0.129032,0.387097
...,...,...,...,...,...,...,...,...,...
12478,13479,6,4,1,1,6,0.571429,0.142857,0.857143
12479,13480,9,3,3,3,8,0.300000,0.300000,0.800000
12480,13481,26,10,11,5,9,0.370370,0.407407,0.333333
12481,13482,38,7,15,16,12,0.179487,0.384615,0.307692


### Engagement Frequency Features

**Idea:** Daily users are more engaged than monthly users.

We convert frequency categories to numeric scores:
- Daily = 30 (highest engagement)
- Weekly = 4
- Monthly = 1 (lowest engagement)

In [169]:
EngagementMetrics.head()

,Logins,Frequency,CustomerID
0,19,Weekly,1001
1,9,Weekly,1002
2,19,Monthly,1003
3,4,Daily,1004
4,12,Weekly,1005


In [170]:
# Map human-readable engagement frequency to numeric weights
# Higher value = more frequent engagement
frequency_map = {
    'Daily': 30,
    'Weekly': 4,
    'Monthly': 1
}

# Aggregate engagement data per customer
EngagementMetrics = EngagementMetrics.groupby('CustomerID').agg({

    # Total number of logins by the customer
    'Logins': 'sum',

    # Keep the first frequency value (assumed consistent per customer)
    'Frequency': 'first'

}).reset_index()


# Convert frequency labels (Daily, Weekly, Monthly) into numeric scores
EngagementMetrics['frequency_score'] = (
    EngagementMetrics['Frequency'].map(frequency_map)
)


# Combine login count and frequency into a single engagement score
# Higher logins + higher frequency = stronger engagement
EngagementMetrics['engagement_score'] = (
    EngagementMetrics['Logins'] * EngagementMetrics['frequency_score']
)


# Keep only the engineered features needed for modeling
EngagementMetrics = EngagementMetrics[
    ['CustomerID', 'Logins', 'frequency_score', 'engagement_score']
]

In [171]:
EngagementMetrics

,CustomerID,Logins,frequency_score,engagement_score
0,1001,19,4,76
1,1002,9,4,36
2,1003,19,1,19
3,1004,4,30,120
4,1005,12,4,48
...,...,...,...,...
12478,13479,22,30,660
12479,13480,25,4,100
12480,13481,9,1,9
12481,13482,2,1,2


### Sentiment & Feedback Features

**Idea:** Negative feedback and low ratings predict churn.

We calculate:
- Average rating
- Comment length (longer = more invested)
- Sentiment flags (positive/negative)

In [172]:
# Aggregate feedback data at the customer level
Feedback = Feedback.groupby('CustomerID').agg({

    # Average rating given by the customer
    'Rating': 'mean',

    # Average length of the customer's comments
    # x is a Series of comments (strings)
    # x.str.len() → length of each comment
    # .mean() → average comment length
    'Comment': lambda x: x.str.len().mean()

}).reset_index()


# Rename columns to reflect engineered features
Feedback.columns = [
    'CustomerID',
    'avg_rating',
    'avg_comment_length'
]


# Normalize rating to a 0–1 range
# Assumes ratings are on a 1–5 scale
Feedback['sentiment_score'] = Feedback['avg_rating'] / 5


# Flag customers with clearly negative feedback
# 1 = negative sentiment, 0 = otherwise
Feedback['is_negative'] = (Feedback['avg_rating'] <= 2).astype(int)


# Flag customers with clearly positive feedback
# 1 = positive sentiment, 0 = otherwise
Feedback['is_positive'] = (Feedback['avg_rating'] >= 4).astype(int)

In [173]:
Feedback

,CustomerID,avg_rating,avg_comment_length,sentiment_score,is_negative,is_positive
0,1001,1.0,96.0,0.2,1,0
1,1002,2.0,108.0,0.4,1,0
2,1003,4.0,72.0,0.8,0,1
3,1004,1.0,78.0,0.2,1,0
4,1005,3.0,99.0,0.6,0,0
...,...,...,...,...,...,...
12478,13479,2.0,37.0,0.4,1,0
12479,13480,3.0,102.0,0.6,0,0
12480,13481,5.0,134.0,1.0,0,1
12481,13482,5.0,113.0,1.0,0,1


## Marketing Response Features

**Idea:** Customers who ignore emails are disengaging.

We calculate:
- Email open rate
- Click rate
- Click-through rate (clicks per open)
- Overall marketing engagement

In [174]:
MarketingCommunication

,Email_Sent,Email_Opened,Email_Clicked,CustomerID
0,2019-10-17,2022-01-12,2022-11-27,1001
1,2019-10-17,2022-01-12,2022-11-27,1001
2,2019-10-17,2022-01-12,2022-11-27,1001
3,2019-10-17,2022-01-12,2022-11-27,1001
4,2019-10-17,2022-01-12,2022-11-27,1001
...,...,...,...,...
68757,2021-02-08,2021-09-18,2022-04-11,13483
68758,2021-02-08,2021-09-18,2022-04-11,13483
68759,2021-02-08,2021-09-18,2022-04-11,13483
68760,2021-02-08,2021-09-18,2022-04-11,13483


In [175]:
# Aggregate marketing communication metrics at the customer level
MarketingCommunication = MarketingCommunication.groupby('CustomerID').agg({

    # Count how many emails were sent to the customer
    'Email_Sent': 'count',

    # Count how many emails were opened by the customer
    'Email_Opened': 'count',

    # Count how many emails were clicked by the customer
    'Email_Clicked': 'count'

}).reset_index()


# Rename columns to meaningful feature names
MarketingCommunication.columns = [
    'CustomerID',       # Customer identifier
    'emails_sent',      # Total emails sent
    'emails_opened',    # Total emails opened
    'emails_clicked'    # Total clicks on emails
]


# Open rate: fraction of emails opened out of emails sent
# +1 in denominator to avoid division by zero
MarketingCommunication['open_rate'] = MarketingCommunication['emails_opened'] / (MarketingCommunication['emails_sent'] + 1)

# Click rate: fraction of emails clicked out of emails sent
MarketingCommunication['click_rate'] = MarketingCommunication['emails_clicked'] / (MarketingCommunication['emails_sent'] + 1)

# Click-through rate (CTR): fraction of clicks among opened emails
# +1 to prevent division by zero
MarketingCommunication['click_through_rate'] = MarketingCommunication['emails_clicked'] / (MarketingCommunication['emails_opened'] + 1)

# Combined engagement metric: interaction strength
# Multiply open rate by click rate as a composite engagement score
MarketingCommunication['marketing_engagement'] = MarketingCommunication['open_rate'] * MarketingCommunication['click_rate']

In [176]:
MarketingCommunication

,CustomerID,emails_sent,emails_opened,emails_clicked,open_rate,click_rate,click_through_rate,marketing_engagement
0,1001,8,8,8,0.888889,0.888889,0.888889,0.790123
1,1002,9,9,9,0.900000,0.900000,0.900000,0.810000
2,1003,8,8,8,0.888889,0.888889,0.888889,0.790123
3,1004,10,10,10,0.909091,0.909091,0.909091,0.826446
4,1005,7,7,7,0.875000,0.875000,0.875000,0.765625
...,...,...,...,...,...,...,...,...
12478,13479,4,4,4,0.800000,0.800000,0.800000,0.640000
12479,13480,7,7,7,0.875000,0.875000,0.875000,0.765625
12480,13481,5,5,5,0.833333,0.833333,0.833333,0.694444
12481,13482,1,1,1,0.500000,0.500000,0.500000,0.250000


### Demographics & Temporal Features

**Idea:** Age, segment, and account age affect churn likelihood.

We create:
- Age groups (Young, Adult, Middle, Senior)
- NPS categories (Detractor, Passive, Promoter)
- Account age in days

In [177]:
# Convert the 'Timestamp' column to datetime format
data['Timestamp'] = pd.to_datetime(data['Timestamp'])

# Calculate account age in days relative to a reference date (2023-01-01)
data['account_age_days'] = (pd.to_datetime('2023-01-01') - data['Timestamp']).dt.days

# Bin customer ages into groups and label them
# 0-25: Young, 26-35: Adult, 36-50: Middle, 51-100: Senior
data['age_group'] = pd.cut(
    data['Age'],
    bins=[0, 25, 35, 50, 100],
    labels=['Young', 'Adult', 'Middle', 'Senior']
)

# Categorize NPS (Net Promoter Score) into three categories
# NPS scale: 0-6 = Detractor, 7-8 = Passive, 9-10 = Promoter
data['nps_category'] = pd.cut(
    data['NPS'],
    bins=[-1, 6, 8, 10],
    labels=['Detractor', 'Passive', 'Promoter']
)

# Create a subset of the dataframe with demographic features for analysis
demographic_features = data[['CustomerID', 'Age', 'Gender', 'Segment', 'NPS',
                             'account_age_days', 'age_group', 'nps_category']]

In [178]:
demographic_features

,CustomerID,Age,Gender,Segment,NPS,account_age_days,age_group,nps_category
0,1001,31,Male,Segment B,3,1069,Adult,Detractor
1,1002,66,Female,Segment C,6,1455,Senior,Detractor
2,1003,36,Female,Segment B,3,1341,Middle,Detractor
3,1004,62,Female,Segment C,1,1033,Senior,Detractor
4,1005,68,Female,Segment C,3,1366,Senior,Detractor
...,...,...,...,...,...,...,...,...
12478,13479,55,Female,Segment A,8,338,Senior,Passive
12479,13480,29,Male,Segment A,7,930,Adult,Passive
12480,13481,38,Male,Segment C,1,809,Middle,Detractor
12481,13482,26,Female,Segment A,0,920,Adult,Detractor


### Merge All Features

In [179]:
# data.merge()

In [180]:
main_df = demographic_features.merge(PurchaseHistory, on='CustomerID', how='left') \
    .merge(SubscriptionDetails, on='CustomerID', how='left') \
    .merge(ServiceInteractions, on='CustomerID', how='left') \
    .merge(PaymentHistory, on='CustomerID', how='left') \
    .merge(WebsiteUsage, on='CustomerID', how='left') \
    .merge(ClickstreamData, on='CustomerID', how='left') \
    .merge(EngagementMetrics, on='CustomerID', how='left') \
    .merge(Feedback, on='CustomerID', how='left') \
    .merge(MarketingCommunication, on='CustomerID', how='left')

# Add the target variable 'ChurnLabel' from the original data to the final merged dataframe
main_df['ChurnLabel'] = data['ChurnLabel']

# Start with the demographic features dataframe
# Merge PurchaseHistory on 'CustomerID', keep all demographic rows (left join) \
# Merge subscription info
# Merge service interactions
# Merge payment history
# Merge website usage data
# Merge clickstream interactions
# Merge engagement metrics (logins, frequency)
# Merge customer feedback info
# Merge marketing email engagement data

In [181]:
main_df

,CustomerID,Age,Gender,Segment,NPS,account_age_days,age_group,nps_category,total_purchase_value,total_frequency,product_diversity,Plan,subscription_duration_days,subscription_age_days,is_active,last_interaction_date,total_interactions,total_late_payments,payment_count,late_payment_rate,payment_risk_score,PageViews,TimeSpent(minutes),engagement_ratio,engagement_intensity,total_actions,add_to_cart_count,search_count,click_count,unique_pages,cart_conversion_rate,search_intensity,page_diversity,Logins,frequency_score,engagement_score,avg_rating,avg_comment_length,sentiment_score,is_negative,is_positive,emails_sent,emails_opened,emails_clicked,open_rate,click_rate,click_through_rate,marketing_engagement,ChurnLabel
0,1001,31,Male,Segment B,3,1069,Adult,Detractor,3994.72,38,7,Express,871,937,0,2021-07-25,4,40,3,10.00,400.00,49,15,0.300000,735,24,8,12,4,13,0.320000,0.480000,0.520000,19,4,76,1.0,96.0,0.2,1,0,8,8,8,0.888889,0.888889,0.888889,0.790123,1
1,1002,66,Female,Segment C,6,1455,Senior,Detractor,2844.35,4,3,Pro,290,529,0,2022-12-13,19,10,3,2.50,25.00,100,9,0.089109,900,24,8,7,9,13,0.320000,0.280000,0.520000,9,4,36,2.0,108.0,0.4,1,0,9,9,9,0.900000,0.900000,0.900000,0.810000,0
2,1003,36,Female,Segment B,3,1341,Middle,Detractor,1866.52,14,3,Essential,319,1184,0,2022-01-04,3,8,3,2.00,16.00,1,97,48.500000,97,12,2,7,3,7,0.153846,0.538462,0.538462,19,1,19,4.0,72.0,0.8,0,1,8,8,8,0.888889,0.888889,0.888889,0.790123,0
3,1004,62,Female,Segment C,1,1033,Senior,Detractor,1378.64,28,5,Smart,803,1083,0,2022-11-10,59,79,3,19.75,1560.25,25,31,1.192308,775,47,15,16,16,14,0.312500,0.333333,0.291667,4,30,120,1.0,78.0,0.2,1,0,10,10,10,0.909091,0.909091,0.909091,0.826446,1
4,1005,68,Female,Segment C,3,1366,Senior,Detractor,2425.05,39,6,Basic,580,633,0,2022-12-19,10,2,3,0.50,1.00,77,51,0.653846,3927,30,17,4,9,12,0.548387,0.129032,0.387097,12,4,48,3.0,99.0,0.6,0,0,7,7,7,0.875000,0.875000,0.875000,0.765625,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12478,13479,55,Female,Segment A,8,338,Senior,Passive,1196.56,14,3,Essential,745,1296,0,2022-10-09,10,3,3,0.75,2.25,70,57,0.802817,3990,6,4,1,1,6,0.571429,0.142857,0.857143,22,30,660,2.0,37.0,0.4,1,0,4,4,4,0.800000,0.800000,0.800000,0.640000,0
12479,13480,29,Male,Segment A,7,930,Adult,Passive,710.57,1,1,Flex,18,22,0,2022-11-05,3,6,3,1.50,9.00,71,66,0.916667,4686,9,3,3,3,8,0.300000,0.300000,0.800000,25,4,100,3.0,102.0,0.6,0,0,7,7,7,0.875000,0.875000,0.875000,0.765625,0
12480,13481,38,Male,Segment C,1,809,Middle,Detractor,5154.42,63,10,Deluxe,20,546,0,2022-12-08,26,83,3,20.75,1722.25,96,1,0.010309,96,26,10,11,5,9,0.370370,0.407407,0.333333,9,1,9,5.0,134.0,1.0,0,1,5,5,5,0.833333,0.833333,0.833333,0.694444,1
12481,13482,26,Female,Segment A,0,920,Adult,Detractor,6055.16,58,9,Gold,484,894,0,2022-11-15,13,67,3,16.75,1122.25,63,2,0.031250,126,38,7,15,16,12,0.179487,0.384615,0.307692,2,1,2,5.0,113.0,1.0,0,1,1,1,1,0.500000,0.500000,0.500000,0.250000,0


#### RFM Analysis

Recency, Frequency and Monetary Analysis = RFM_score

Based on the rfm_score we are segmenting our customers

In [182]:
# RFM Analysis
currentDate = main_df['last_interaction_date'].max()

main_df['Recency'] = (currentDate - main_df['last_interaction_date'])
main_df['Frequency'] = main_df['total_frequency'] * main_df['Logins']
main_df['Monetary'] = main_df['total_purchase_value']

# Set bins [1, 2, 3, 4, 5] for R, F, M scores
r_bins = range(5, 0, -1)
f_bins = range(1, 6)
m_bins = range(1, 6)

main_df['r_score'] = pd.qcut(main_df['Recency'], 5, labels=r_bins)
main_df['f_score'] = pd.qcut(main_df['Frequency'], 5, labels=f_bins)
main_df['m_score'] = pd.qcut(main_df['Monetary'], 5, labels=m_bins)

main_df['rfm_score'] = main_df['r_score'].astype(int) + main_df['f_score'].astype(int) + main_df['m_score'].astype(int)
main_df[['Recency', 'Frequency', 'Monetary', 'r_score', 'f_score', 'm_score', 'rfm_score']]

# Customer Segmentation based on RFM Score
def rfm_segment(rfm_score):
    if rfm_score >= 13:
        return 'Champions'
    elif 10 <= rfm_score < 13:
        return 'Loyal Customers'
    elif 7 <= rfm_score < 10:
        return 'Neutral Customers'
    elif 5 <= rfm_score < 7:
        return 'Needs Attention'
    else:
        return 'At Risk'
    
main_df['rfm_segment'] = main_df['rfm_score'].apply(rfm_segment)
main_df[['rfm_score', 'rfm_segment']]

,rfm_score,rfm_segment
0,9,Neutral Customers
1,9,Neutral Customers
2,6,Needs Attention
3,7,Neutral Customers
4,11,Loyal Customers
...,...,...
12478,8,Neutral Customers
12479,5,Needs Attention
12480,13,Champions
12481,10,Loyal Customers


### Encoding / Feature Encoding

In [183]:
main_df.dtypes

CustomerID                              int64
Age                                     int64
Gender                                 object
Segment                                object
NPS                                     int64
account_age_days                        int64
age_group                            category
nps_category                         category
total_purchase_value                  float64
total_frequency                         int64
product_diversity                       int64
Plan                                   object
subscription_duration_days              int64
subscription_age_days                   int64
is_active                               int64
last_interaction_date          datetime64[ns]
total_interactions                      int64
total_late_payments                     int64
payment_count                           int64
late_payment_rate                     float64
payment_risk_score                    float64
PageViews                         

In [184]:
for col in main_df.select_dtypes(include=['object', 'category']).columns.tolist():
    print(f'{col}: {main_df[col].nunique()}')

Gender: 2
Segment: 3
age_group: 4
nps_category: 3
Plan: 20
r_score: 5
f_score: 5
m_score: 5
rfm_segment: 5



Target encoding: Replace a category with the mean of the target variable for that category. \
One-hot encoding: Convert each category into a separate binary (0/1) feature. \
Label encoding: Assign each category a unique integer value.


#### Type of categorical variable

- Nominal CV 
- Ordinal CV

In [190]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

nominal_column = ['Gender', 'Segment', 'nps_category']
ordinal_column = ['age_group', 'rfm_segment']
cardinal_column = ['Plan']

ohe = OneHotEncoder(sparse_output=False)

for col in nominal_column:
    transformed_col = ohe.fit_transform(main_df[[col]])
    # print(transformed_col)
    transformed_df = pd.DataFrame(transformed_col, columns = ohe.get_feature_names_out([col]))
    main_df = pd.concat([main_df, transformed_df], axis=1)
    # main_df.drop(columns=[col], inplace=True)

# .fit is used to train the encoder on the unique values in the column while 
# .transform is used to convert the actual data into the encoded 
# pd.concat is used to concatenate the new encoded columns to the original dataframe
# inplace=True is used to modify the original dataframe directly without creating a copy


In [186]:
main_df

,CustomerID,Age,Gender,Segment,NPS,account_age_days,age_group,nps_category,total_purchase_value,total_frequency,product_diversity,Plan,subscription_duration_days,subscription_age_days,is_active,last_interaction_date,total_interactions,total_late_payments,payment_count,late_payment_rate,payment_risk_score,PageViews,TimeSpent(minutes),engagement_ratio,engagement_intensity,total_actions,add_to_cart_count,search_count,click_count,unique_pages,cart_conversion_rate,search_intensity,page_diversity,Logins,frequency_score,engagement_score,avg_rating,avg_comment_length,sentiment_score,is_negative,is_positive,emails_sent,emails_opened,emails_clicked,open_rate,click_rate,click_through_rate,marketing_engagement,ChurnLabel,Recency,Frequency,Monetary,r_score,f_score,m_score,rfm_score,rfm_segment,Gender_Female,Gender_Male,Segment_Segment A,Segment_Segment B,Segment_Segment C,nps_category_Detractor,nps_category_Passive,nps_category_Promoter
0,1001,31,Male,Segment B,3,1069,Adult,Detractor,3994.72,38,7,Express,871,937,0,2021-07-25,4,40,3,10.00,400.00,49,15,0.300000,735,24,8,12,4,13,0.320000,0.480000,0.520000,19,4,76,1.0,96.0,0.2,1,0,8,8,8,0.888889,0.888889,0.888889,0.790123,1,523 days,722,3994.72,1,4,4,9,Neutral Customers,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,1002,66,Female,Segment C,6,1455,Senior,Detractor,2844.35,4,3,Pro,290,529,0,2022-12-13,19,10,3,2.50,25.00,100,9,0.089109,900,24,8,7,9,13,0.320000,0.280000,0.520000,9,4,36,2.0,108.0,0.4,1,0,9,9,9,0.900000,0.900000,0.900000,0.810000,0,17 days,36,2844.35,5,1,3,9,Neutral Customers,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,1003,36,Female,Segment B,3,1341,Middle,Detractor,1866.52,14,3,Essential,319,1184,0,2022-01-04,3,8,3,2.00,16.00,1,97,48.500000,97,12,2,7,3,7,0.153846,0.538462,0.538462,19,1,19,4.0,72.0,0.8,0,1,8,8,8,0.888889,0.888889,0.888889,0.790123,0,360 days,266,1866.52,1,3,2,6,Needs Attention,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,1004,62,Female,Segment C,1,1033,Senior,Detractor,1378.64,28,5,Smart,803,1083,0,2022-11-10,59,79,3,19.75,1560.25,25,31,1.192308,775,47,15,16,16,14,0.312500,0.333333,0.291667,4,30,120,1.0,78.0,0.2,1,0,10,10,10,0.909091,0.909091,0.909091,0.826446,1,50 days,112,1378.64,3,2,2,7,Neutral Customers,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
4,1005,68,Female,Segment C,3,1366,Senior,Detractor,2425.05,39,6,Basic,580,633,0,2022-12-19,10,2,3,0.50,1.00,77,51,0.653846,3927,30,17,4,9,12,0.548387,0.129032,0.387097,12,4,48,3.0,99.0,0.6,0,0,7,7,7,0.875000,0.875000,0.875000,0.765625,0,11 days,468,2425.05,5,3,3,11,Loyal Customers,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12478,13479,55,Female,Segment A,8,338,Senior,Passive,1196.56,14,3,Essential,745,1296,0,2022-10-09,10,3,3,0.75,2.25,70,57,0.802817,3990,6,4,1,1,6,0.571429,0.142857,0.857143,22,30,660,2.0,37.0,0.4,1,0,4,4,4,0.800000,0.800000,0.800000,0.640000,0,82 days,308,1196.56,3,3,2,8,Neutral Customers,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
12479,13480,29,Male,Segment A,7,930,Adult,Passive,710.57,1,1,Flex,18,22,0,2022-11-05,3,6,3,1.50,9.00,71,66,0.916667,4686,9,3,3,3,8,0.300000,0.300000,0.800000,25,4,100,3.0,102.0,0.6,0,0,7,7,7,0.875000,0.875000,0.875000,0.765625,0,55 days,25,710.57,3,1,1,5,Needs Attention,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
12480,13481,38,Male,Segment C,1,809,Middle,Detractor,5154.42,63,10,Deluxe,20,546,0,2022-12-08,26,83,3,20.75,1722.25,96,1,0.010309,96,26,10,11,5,9,0.370370,0.407407,0.333333,9,1,9,5.0,134.0,1.0,0,1,5,5,5,0.833333,0.833333,0.833333,0.694444,1,22 days,567,5154.42,4,4,5,13,Champions,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0
12481,13482,26,Female,Segment A,0,920,Adult,Detractor,6055.16,58,9,Gold,484,894,0,2022-11-15,13,67,3,16.75,1122.25,63,2,0.031250,126,38,7,15,16,12,0.179487,0.384615,0.307692,2,1,2,5.0,113.0,1.0,0,1,1,1,1,0.500000,0.500000,0.500000,0.250000,0,45 days,116,6055.16,3,2,5,10,Loyal Cu

In [187]:
le = LabelEncoder()
for col in ordinal_column:
    main_df[col] = le.fit_transform(main_df[col])

In [189]:
for col in cardinal_column:
    main_df[col] = main_df[col].groupby(col)['total_purchase_value'].transform('mean')

KeyError: 'Plan'